# PEGASUS SAMSum Summarization
Fine-tuning PEGASUS on SAMSum Dataset

In [ ]:
!pip install transformers datasets sentencepiece accelerate evaluate rouge_score sacrebleu py7zr

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from transformers import DataCollatorForSeq2Seq, TrainingArguments, Trainer
import evaluate

device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
model_name = "google/pegasus-xsum"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

In [ ]:
dataset = load_dataset("knkarthick/samsum")
dataset

In [ ]:
def convert_ex_to_features(batch):
    input_enc = tokenizer(batch['dialogue'], max_length=512, truncation=True)
    target_enc = tokenizer(text_target=batch['summary'], max_length=128, truncation=True)
    return {
        'input_ids': input_enc['input_ids'],
        'attention_mask': input_enc['attention_mask'],
        'labels': target_enc['input_ids']
    }

tokenized_data = dataset.map(convert_ex_to_features, batched=True)

In [ ]:
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

In [ ]:
training_args = TrainingArguments(
    output_dir="pegasus-samsum",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    warmup_steps=500,
    logging_steps=50,
    save_steps=5000,
    evaluation_strategy="steps",
    eval_steps=500,
    report_to="none"
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    tokenizer=tokenizer,
    data_collator=data_collator,
    train_dataset=tokenized_data['train'],
    eval_dataset=tokenized_data['validation']
)

In [ ]:
trainer.train()